# GPT-OSS contextual versus isolated detail representations across age

This notebook uses only **actual-age summaries**. It asks whether an original detail's GPT-OSS representation changes when it is encoded inside its story rather than independently, and whether that contextualization varies with source age. The analyses proceed from all details to retained-versus-omitted details and then central-versus-peripheral details.

## Definitions and matched readout

The primary outcome is

$$\text{contextualization shift}_i = 1-\cos(z^{ctx}_i,z^{iso}_i).$$

Larger values mean that the same detail is represented more differently inside its original story than when encoded alone. Both vectors use final-layer **mean token pooling**: contextual `mean_control_vectors` and isolated `mean_vectors`. This avoids the last-token-versus-EOS readout mismatch in the earlier notebook.

- **Retained:** MPNet independently labels the original detail as retained in the actual summary (`retained_cosine`).
- **Omitted:** the original detail is not retained under that rule.
- **Centrality:** mean cosine similarity between a contextual original-detail vector and all other contextual detail vectors in the same full story.
- **Central:** at or above the within-story median centrality; **peripheral:** below it.

The observational unit for inference is the story or summary-level mean, not each detail as an independent replicate.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

ROOT = Path(os.environ.get('STORY_FEATURES_ROOT', Path.cwd())).resolve()
if not (ROOT / 'mechanistic_interpretability').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.analyze_gpt_oss_contextualization_age import (
    AGE_CENTER, aggregate_two_rows, fixed_effect_table, load_analysis_data,
    plot_group_trajectories, slope_tests,
)
CONTEXT_DIR = Path(os.environ.get(
    'GPT_OSS_CONTEXT_DIR',
    ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/gpt_oss_20b_global_current',
))
ISOLATED_DIR = Path(os.environ.get(
    'GPT_OSS_ISOLATED_DIR',
    ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/gpt_oss_20b_detail_controls',
))
MPNET_DIR = Path(os.environ.get(
    'MPNET_ALIGNMENT_DIR',
    ROOT / 'mechanistic_interpretability/outputs/13_structured_detail_alignment',
))

details, coverage_audit = load_analysis_data(CONTEXT_DIR, ISOLATED_DIR, MPNET_DIR)
display(coverage_audit)
display(details.groupby('retention_status', observed=True).agg(
    details=('original_detail_index', 'size'),
    stories=('story_key', 'nunique'),
    summaries=('summary_key', 'nunique'),
    mean_shift=('contextualization_shift', 'mean'),
))
print('Source-age range:', details.source_age.min(), 'to', details.source_age.max())

## Control 0 — Are the conclusions robust to the representation readout?

Before comparing contextual and isolated representations, check whether alternative pooling choices produce similar vectors for the same detail. Existing files support two comparisons:

1. **Contextual:** mean of the detail-token states versus the last detail-token state at the same native story position.
2. **Isolated:** mean of the independently encoded detail-token states versus its EOS state.

The contextual extractor did not insert EOS after every detail prefix, so comparison 1 is not literally contextual mean versus contextual EOS. An exact contextual-EOS control would require new GPT-OSS forward passes. As a descriptive practical-agreement criterion, the table reports the proportion of cosine similarities at least 0.95, together with the lower tail. Story-level regressions test whether readout agreement systematically changes with source age.

In [ ]:
control_details = (details.sort_values(['story_key', 'original_detail_index'])
                   .drop_duplicates(['story_key', 'original_detail_index']))
control_metrics = {
    'contextual mean vs last token': 'context_mean_last_cosine',
    'isolated mean vs EOS': 'isolated_mean_eos_cosine',
}
agreement_rows = []
age_rows = []
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (label, column) in zip(axes, control_metrics.items()):
    values = control_details[column].dropna().to_numpy(float)
    agreement_rows.append({
        'comparison': label, 'details': len(values), 'mean_cosine': values.mean(),
        'median_cosine': np.median(values), 'p05_cosine': np.quantile(values, .05),
        'proportion_cosine_ge_0.95': np.mean(values >= .95),
    })
    story_control = control_details.groupby(
        ['story_key', 'source_age'], observed=True, as_index=False
    ).agg(mean_agreement=(column, 'mean'))
    story_control['source_age_centered'] = story_control.source_age - AGE_CENTER
    control_fit = smf.ols(
        'mean_agreement ~ source_age_centered', story_control
    ).fit(cov_type='HC3')
    age_term = fixed_effect_table(control_fit).query("term == 'source_age_centered'").iloc[0]
    age_rows.append({'comparison': label, **age_term.to_dict()})
    ax.hist(values, bins=60, density=True, alpha=.75)
    ax.axvline(.95, color='black', linestyle='--', linewidth=1, label='0.95 criterion')
    ax.set(title=label, xlabel='Cosine agreement', ylabel='Density')
    ax.legend()
display(pd.DataFrame(agreement_rows).round(5))
print('Does pooling/readout agreement change with source age?')
display(pd.DataFrame(age_rows)[[
    'comparison', 'estimate', 'std_error', 'p_value', 'ci95_low', 'ci95_high', 'significant_05'
]].round(5))
plt.tight_layout()
plt.show()

## RQ1 — Across all original details, does contextualization change with source age?

First deduplicate original details across any repeated actual-summary rows, average contextualization shift within each story, and fit `story_mean_shift ~ source_age` with HC3 robust standard errors. The age coefficient is the change in mean contextualization shift per additional year.

In [ ]:
unique_original_details = (details.sort_values(['story_key', 'original_detail_index'])
                           .drop_duplicates(['story_key', 'original_detail_index']))
story_means = unique_original_details.groupby(
    ['story_key', 'source_age'], observed=True, as_index=False
).agg(
    mean_contextualization_shift=('contextualization_shift', 'mean'),
    details=('original_detail_index', 'size'),
)
story_means['source_age_centered'] = story_means.source_age - AGE_CENTER
overall_fit = smf.ols(
    'mean_contextualization_shift ~ source_age_centered', data=story_means
).fit(cov_type='HC3')
print(overall_fit.summary())
display(fixed_effect_table(overall_fit).round(5))

age_grid = np.linspace(story_means.source_age.min(), story_means.source_age.max(), 100)
grid = pd.DataFrame({'source_age': age_grid, 'source_age_centered': age_grid - AGE_CENTER})
prediction = overall_fit.get_prediction(grid).summary_frame()
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(story_means.source_age, story_means.mean_contextualization_shift, alpha=.10, s=12)
ax.plot(age_grid, prediction['mean'], color='black', linewidth=2)
ax.fill_between(age_grid, prediction['mean_ci_lower'], prediction['mean_ci_upper'], color='black', alpha=.15)
ax.set(xlabel='Source age', ylabel='Story mean contextualization shift (1 − cosine)',
       title='All original details in actual-summary stories')
plt.show()

## RQ2 — Does the age trajectory differ for details retained versus omitted in the actual summary?

Each usable actual summary contributes exactly two rows: its mean shift across retained original details and its mean shift across omitted original details. Fit

```r
mean_shift ~ source_age * retention_status + (1 | story)
```

with retained as the reference. Derived tests report the retained slope, omitted slope, and their difference.

In [ ]:
retention_rows = aggregate_two_rows(
    details, 'retention_status',
    ['story_key', 'summary_key', 'source_age', 'source_age_centered'],
)
retention_rows['retention_status'] = pd.Categorical(
    retention_rows.retention_status, categories=['retained', 'omitted'], ordered=True
)
display(retention_rows.groupby('retention_status', observed=True).agg(
    rows=('mean_contextualization_shift', 'size'),
    stories=('story_key', 'nunique'),
    observed_mean=('mean_contextualization_shift', 'mean'),
))
retention_formula = (
    'mean_contextualization_shift ~ source_age_centered * '
    'C(retention_status, Treatment(reference="retained"))'
)
retention_fit = smf.mixedlm(
    retention_formula, retention_rows, groups=retention_rows.story_key, re_formula='1'
).fit(reml=False, method='lbfgs')
print(retention_fit.summary())
display(fixed_effect_table(retention_fit).round(5))
retention_slopes = slope_tests(
    retention_fit, retention_formula, 'retention_status', ['retained', 'omitted']
)
display(retention_slopes.round(5))
plot_group_trajectories(
    retention_fit, retention_formula, 'retention_status', ['retained', 'omitted'],
    retention_rows, 'Contextualization by actual-summary retention and source age',
)

## RQ3 — Does the age trajectory differ for central versus peripheral original details?

Deduplicate original details, then give each story two rows: mean shift for central details and mean shift for peripheral details. Fit

```r
mean_shift ~ source_age * centrality_group + (1 | story)
```

Derived tests report the central slope, peripheral slope, and their difference. Centrality is calculated from contextual vectors, so this analysis is associational rather than an independent causal test of centrality.

In [ ]:
centrality_rows = aggregate_two_rows(
    unique_original_details, 'centrality_group',
    ['story_key', 'source_age', 'source_age_centered'],
)
centrality_rows['centrality_group'] = pd.Categorical(
    centrality_rows.centrality_group, categories=['central', 'peripheral'], ordered=True
)
display(centrality_rows.groupby('centrality_group', observed=True).agg(
    rows=('mean_contextualization_shift', 'size'),
    stories=('story_key', 'nunique'),
    observed_mean=('mean_contextualization_shift', 'mean'),
))
centrality_formula = (
    'mean_contextualization_shift ~ source_age_centered * '
    'C(centrality_group, Treatment(reference="central"))'
)
centrality_fit = smf.mixedlm(
    centrality_formula, centrality_rows, groups=centrality_rows.story_key, re_formula='1'
).fit(reml=False, method='lbfgs')
print(centrality_fit.summary())
display(fixed_effect_table(centrality_fit).round(5))
centrality_slopes = slope_tests(
    centrality_fit, centrality_formula, 'centrality_group', ['central', 'peripheral']
)
display(centrality_slopes.round(5))
plot_group_trajectories(
    centrality_fit, centrality_formula, 'centrality_group', ['central', 'peripheral'],
    centrality_rows, 'Contextualization by original-detail centrality and source age',
)

## RQ4 — How are centrality and actual-summary retention related?

This descriptive table checks overlap between the two partitions. It prevents a retained-versus-omitted result from being mistaken for an independent centrality result.

In [ ]:
overlap = details.groupby(['centrality_group', 'retention_status'], observed=True).agg(
    detail_observations=('original_detail_index', 'size'),
    stories=('story_key', 'nunique'),
    mean_shift=('contextualization_shift', 'mean'),
).reset_index()
display(overlap)
retention_by_centrality = (details.groupby('centrality_group', observed=True)['retained']
                           .agg(['size', 'mean']).rename(columns={'mean': 'retention_rate'}))
display(retention_by_centrality)

## Interpretation boundaries

- The outcome measures representational change, not information loss or model surprise.
- Mean-to-mean pooling controls the readout mismatch, but contextual and isolated inputs necessarily contain different surrounding text.
- MPNet defines retention independently of GPT-OSS; it is threshold-based and can contain matching errors.
- Centrality and contextualization both use contextual GPT-OSS vectors, so their association is not fully independent.
- The models estimate associations with source age in actual-age summaries; they do not establish that age causes the representational differences.